# TSUNAMI quantitative figure export

This one-off companion notebook recreates publication-facing descriptive
figures from the analysis-ready CSV files. It does not perform or alter the
inferential analyses in `Tsunami_quantitative_analysis.ipynb`. Precision
Task figures use the same embedded failed-manipulation exclusion flags as
the primary analysis. Every figure is exported as SVG to `./figures`.


In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

DATA_DIR = Path('./input')
FIGURES_DIR = Path('./figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
for previous_figure in FIGURES_DIR.glob('*.svg'):
    previous_figure.unlink()

METHODS = ('baseline', 'map', 'tsunami')
LONG_RANGE_METHODS = ('map', 'tsunami')
ROUTES = ('A', 'B', 'C')
METHOD_LABELS = {'baseline': 'Baseline', 'map': 'Minimap', 'tsunami': 'Tsunami'}
METHOD_COLORS = {'baseline': '#7f7f7f', 'map': '#dd8452', 'tsunami': '#4c72b0'}
DISTANCE_LABELS = {50: 'Short (50 m)', 100: 'Medium (100 m)', 200: 'Long (200 m)'}
PRECISION_TARGET_X = {50: 81.7, 100: 31.7, 200: -68.3}
ROUTE_LENGTHS_M = {'A': 2589.5290707671866, 'B': 2411.0317353962837,
                   'C': 2580.0081858076164}
METHOD_SEQUENCES = (
    ('baseline', 'map', 'tsunami'), ('map', 'tsunami', 'baseline'),
    ('tsunami', 'baseline', 'map'), ('baseline', 'tsunami', 'map'),
    ('map', 'baseline', 'tsunami'), ('tsunami', 'map', 'baseline'),
)
COUNTERBALANCE_SLOT_OVERRIDES = {19: 18, 20: 15, 21: 12, 22: 7, 23: 4, 24: 1}

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({
    'figure.dpi': 120, 'savefig.dpi': 300, 'svg.fonttype': 'none',
    'axes.titleweight': 'bold', 'axes.spines.top': False,
    'axes.spines.right': False,
})

def participant_uid(location, participant):
    p = participant.astype(str).str.replace(r'\.0$', '', regex=True).str.upper()
    p = p.where(p.str.startswith('P'), 'P' + p.str.zfill(2))
    return location.astype(str).str.upper() + '_' + p

def method_sequence(participant_id):
    number = int(float(str(participant_id).upper().replace('P', '')))
    slot = number if 1 <= number <= 18 else COUNTERBALANCE_SLOT_OVERRIDES[number]
    return METHOD_SEQUENCES[(slot - 1) // 3]

EXPORTED_FIGURES = []
def save_svg(fig, filename):
    path = FIGURES_DIR / filename
    fig.savefig(path, format='svg', bbox_inches='tight')
    EXPORTED_FIGURES.append(path)
    if 'agg' not in plt.get_backend().lower():
        plt.show()
    plt.close(fig)
    print(f'Exported: {path}')

def add_box_and_points(ax, data, x, y, order, palette=METHOD_COLORS,
                       hue=None, hue_order=None):
    sns.boxplot(
        data=data, x=x, y=y, order=order, hue=hue or x,
        hue_order=hue_order or order, palette=palette, showfliers=False,
        width=.62, linewidth=1, dodge=hue is not None, ax=ax,
    )
    sns.stripplot(
        data=data, x=x, y=y, order=order, hue=hue, hue_order=hue_order,
        palette=palette if hue else None, color=None if hue else 'black',
        dodge=hue is not None, jitter=.15, alpha=.43, size=3,
        linewidth=.25, edgecolor='white', legend=False, ax=ax,
    )
    if ax.get_legend() is not None:
        ax.get_legend().remove()
    ax.grid(axis='y', alpha=.22)

def target_scatter(ax, data, title, limit=None):
    for method in data['Method'].dropna().unique():
        subset = data.loc[data['Method'].eq(method)]
        ax.scatter(
            subset['OffsetX'], subset['OffsetZ'], s=16, alpha=.48,
            color=METHOD_COLORS[method], label=METHOD_LABELS[method],
            edgecolors='none',
        )
    ax.scatter([0], [0], marker='+', s=110, linewidth=2, color='black', label='Target')
    for radius in (1, 2, 5):
        ax.add_patch(plt.Circle((0, 0), radius, fill=False, color='0.55',
                                linewidth=.65, linestyle='--', alpha=.65))
    if limit is None:
        extent = np.nanquantile(np.abs(data[['OffsetX', 'OffsetZ']]), .99)
        limit = max(6, float(extent) * 1.08)
    ax.set_xlim(-limit, limit); ax.set_ylim(-limit, limit)
    ax.set_aspect('equal', adjustable='box')
    ax.axhline(0, color='0.75', linewidth=.6); ax.axvline(0, color='0.75', linewidth=.6)
    ax.set_xlabel('Target-centred X offset (m)')
    ax.set_ylabel('Target-centred Z offset (m)')
    ax.set_title(title)


## Load and prepare the analysis-ready data


In [2]:
def read(name):
    path = DATA_DIR / name
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)

city = read('city_race_aggregated.csv')
city['ParticipantUID'] = participant_uid(city['Location'], city['ParticipantID'])
city['Method'] = city['Method'].str.lower()
city['PathID'] = city['PathID'].str.upper()
city['IdealRouteLength'] = city['PathID'].map(ROUTE_LENGTHS_M)
city['PathEfficiency'] = city['IdealRouteLength'] / city['TotalTraversedDistance']

precision = read('precision.csv')
precision['Distance'] = pd.to_numeric(precision['Distance']).astype(int)
for column in ('RecommendedExclude', 'RepeatedFailureTrialExclude'):
    precision[column] = precision[column].astype(str).str.strip().str.lower().isin(('true', '1', 'yes'))
precision['ExcludeFailedManipulation'] = (
    precision['RecommendedExclude'] | precision['RepeatedFailureTrialExclude']
)
precision_clean = precision.loc[~precision['ExcludeFailedManipulation']].copy()
precision_clean['TargetX'] = precision_clean['Distance'].map(PRECISION_TARGET_X)
precision_clean['OffsetX'] = precision_clean['LandingPos_x'] - precision_clean['TargetX']
precision_clean['OffsetZ'] = precision_clean['LandingPos_z']

city_precision = read('city_race_precision_200_to_500m.csv')
city_precision['Method'] = city_precision['Method'].str.lower()
city_precision['DistanceBandAnalysis'] = pd.cut(
    city_precision['TargetDistance2D'], [200, 350, 500], right=False,
    labels=['200–350 m', '350–500 m'],
)
city_precision['OffsetX'] = city_precision['LandingOffset_x']
city_precision['OffsetZ'] = city_precision['LandingOffset_z']

episodes = read('city_race_checkpoint_approach_episodes.csv')
episodes['Method'] = episodes['Method'].str.lower()
episodes['DistanceBandAnalysis'] = pd.cut(
    episodes['InitialTargetDistance2D'], [200, 350, 500], right=False,
    labels=['200–350 m', '350–500 m'],
)
episodes['OffsetX'] = episodes['FirstLandingPos_x'] - episodes['TargetPos_x']
episodes['OffsetZ'] = episodes['FirstLandingPos_z'] - episodes['TargetPos_z']

orientation = read('city_race_orientation_events.csv')
orientation['Method'] = orientation['Method'].str.lower()

SPATIAL_COLUMNS = {
    'position_awareness': 'After teleporting, I remained aware of my position in the environment.',
    'direction_awareness': 'After teleporting, I remained aware of the direction I was facing.',
    'reorientation_effort': 'After teleporting, I needed additional effort to reorient myself.',
    'environmental_continuity': 'Movement between locations felt spatially continuous.',
    'environment_distance': 'I was able to accurately estimate distances between locations while navigating.',
    'destination_distance': 'I was able to accurately estimate the distance to my intended teleport destination.',
    'easy_to_learn': 'I found this locomotion technique easy to learn.',
    'intuitive': 'I found this locomotion technique intuitive to use.',
}
spatial_parts = []
for method in METHODS:
    frame = read(f'questionnaire_{"minimap" if method == "map" else method}.csv')
    out = pd.DataFrame({
        'ParticipantUID': participant_uid(frame['Location'], frame['Participant ID']),
        'Method': method,
    })
    for short, source_column in SPATIAL_COLUMNS.items():
        out[short] = pd.to_numeric(frame[source_column], errors='coerce')
    out['reorientation_effort_reversed'] = 7 - out['reorientation_effort']
    out['SpatialAwarenessComposite'] = out[[
        'position_awareness', 'direction_awareness', 'reorientation_effort_reversed'
    ]].mean(axis=1)
    out['UsabilityComposite'] = out[['easy_to_learn', 'intuitive']].mean(axis=1)
    spatial_parts.append(out)
spatial = pd.concat(spatial_parts, ignore_index=True)

tlx = read('questionnaire_raw-tlx.csv')
tlx['ParticipantUID'] = participant_uid(tlx['Location'], tlx['Participant ID'])
tlx_long = tlx.melt(
    id_vars=['ParticipantUID'],
    value_vars=['Baseline TLX', 'Minimap TLX', 'Tsunami TLX'],
    var_name='MethodLabel', value_name='RAWTLX',
)
tlx_long['Method'] = tlx_long['MethodLabel'].map({
    'Baseline TLX': 'baseline', 'Minimap TLX': 'map', 'Tsunami TLX': 'tsunami'
})

csq = read('questionnaire_cybersickness.csv')
csq_rows, csq_time_rows = [], []
for _, row in csq.iterrows():
    uid = participant_uid(pd.Series([row['Location']]), pd.Series([row['Participant ID']])).iloc[0]
    for measurement in range(4):
        csq_time_rows.append({
            'ParticipantUID': uid, 'Measurement': measurement,
            'CSQTotal': float(row[f'CSQ-{measurement} Total']),
        })
    previous = float(row['CSQ-0 Total'])
    for session, method in enumerate(method_sequence(row['Participant ID']), start=1):
        total = float(row[f'CSQ-{session} Total'])
        csq_rows.append({'ParticipantUID': uid, 'Method': method,
                         'CSQTotalChange': total - previous})
        previous = total
csq_long = pd.DataFrame(csq_rows)
csq_time = pd.DataFrame(csq_time_rows)

preferences = read('questionnaire_preferences.csv')
print({
    'city_rows': len(city), 'precision_retained': len(precision_clean),
    'precision_excluded': int(precision['ExcludeFailedManipulation'].sum()),
    'city_direct_landings': len(city_precision), 'checkpoint_episodes': len(episodes),
    'orientation_events': len(orientation), 'participants': spatial['ParticipantUID'].nunique(),
})


{'city_rows': 378, 'precision_retained': 1115, 'precision_excluded': 19, 'city_direct_landings': 870, 'checkpoint_episodes': 1492, 'orientation_events': 2231, 'participants': 42}


## RQ1 — Practical viability


In [3]:
fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.1))
for ax, outcome, title in zip(
    axes, ('NetPathTime', 'TotalPathTime'),
    ('Net time (banner delay excluded)', 'Total time (banner delay included)'),
):
    sns.boxplot(data=city, x='PathID', y=outcome, hue='Method', order=ROUTES,
                hue_order=METHODS, palette=METHOD_COLORS, showfliers=False,
                width=.72, ax=ax)
    sns.stripplot(data=city, x='PathID', y=outcome, hue='Method', order=ROUTES,
                  hue_order=METHODS, palette=METHOD_COLORS, dodge=True,
                  jitter=.13, alpha=.5, size=2.6, legend=False, ax=ax)
    ax.set(title=title, xlabel='Route', ylabel='Time (s)')
handles, _ = axes[1].get_legend_handles_labels()
axes[0].get_legend().remove()
axes[1].get_legend().remove()
fig.legend(handles[:3], [METHOD_LABELS[m] for m in METHODS], title='Method', ncol=3,
           loc='lower center', bbox_to_anchor=(.5, -.01))
fig.tight_layout(rect=(0, .13, 1, 1))
save_svg(fig, 'rq1_city_race_completion_times.svg')


Exported: figures/rq1_city_race_completion_times.svg


In [4]:
city_participant = city.groupby(['ParticipantUID', 'Method'], as_index=False).agg(
    TotalTeleports=('TotalTeleports', 'mean'), PathEfficiency=('PathEfficiency', 'mean')
)
fig, axes = plt.subplots(1, 2, figsize=(8.6, 3.8))
for ax, outcome, title, ylabel in (
    (axes[0], 'TotalTeleports', 'Interaction effort', 'Teleport events'),
    (axes[1], 'PathEfficiency', 'Path efficiency', 'Ideal / traversed distance'),
):
    add_box_and_points(ax, city_participant, 'Method', outcome, METHODS)
    ax.set_xticks(range(3), [METHOD_LABELS[m] for m in METHODS])
    ax.set(title=title, xlabel='', ylabel=ylabel)
fig.tight_layout()
save_svg(fig, 'rq1_interaction_efficiency.svg')


Exported: figures/rq1_interaction_efficiency.svg


In [5]:
fig, axes = plt.subplots(1, 3, figsize=(11.2, 3.8), sharey=False)
for ax, distance in zip(axes, (50, 100, 200)):
    subset = precision_clean.loc[precision_clean['Distance'].eq(distance)]
    add_box_and_points(ax, subset, 'Method', 'AimingError', METHODS)
    ax.set_xticks(range(3), [METHOD_LABELS[m] for m in METHODS], rotation=15)
    ax.set(title=DISTANCE_LABELS[distance], xlabel='', ylabel='Landing error (m)')
fig.tight_layout()
save_svg(fig, 'rq1_precision_landing_error.svg')

fig, axes = plt.subplots(2, 3, figsize=(11.2, 6.8), sharex=True)
for column, distance in enumerate((50, 100, 200)):
    subset = precision_clean.loc[precision_clean['Distance'].eq(distance)]
    for row, outcome, ylabel in ((0, 'HoldTime', 'Aim-to-teleport time (s)'),
                                 (1, 'CompletionTime', 'Completion time (s)')):
        ax = axes[row, column]
        add_box_and_points(ax, subset, 'Method', outcome, METHODS)
        ax.set_xticks(range(3), [METHOD_LABELS[m] for m in METHODS], rotation=15)
        ax.set(title=DISTANCE_LABELS[distance] if row == 0 else '', xlabel='', ylabel=ylabel)
fig.tight_layout()
save_svg(fig, 'rq1_precision_timing.svg')

fig, ax = plt.subplots(figsize=(5.8, 4.2))
target_scatter(ax, precision_clean.loc[precision_clean['Distance'].eq(200)], '200 m target')
ax.legend(frameon=True, fontsize=8, ncol=1, loc='center left',
          bbox_to_anchor=(1.03, .5))
fig.subplots_adjust(left=.14, right=.72, bottom=.16, top=.90)
save_svg(fig, 'rq1_precision_200m_scatter.svg')


Exported: figures/rq1_precision_landing_error.svg
Exported: figures/rq1_precision_timing.svg
Exported: figures/rq1_precision_200m_scatter.svg


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.8), sharey=True)
for ax, band in zip(axes, ('200–350 m', '350–500 m')):
    subset = city_precision.loc[city_precision['DistanceBandAnalysis'].eq(band)]
    add_box_and_points(ax, subset, 'Method', 'LandingError2D', LONG_RANGE_METHODS)
    ax.set_xticks(range(2), [METHOD_LABELS[m] for m in LONG_RANGE_METHODS])
    ax.set(title=band, xlabel='', ylabel='Direct landing error (m)')
fig.tight_layout()
save_svg(fig, 'rq1_city_direct_landing_error.svg')

fig, axes = plt.subplots(2, 2, figsize=(8.1, 7.4), sharex=True, sharey=True)
common_limit = max(10, float(np.nanquantile(np.abs(city_precision[['OffsetX', 'OffsetZ']]), .99)) * 1.08)
for row, band in enumerate(('200–350 m', '350–500 m')):
    for column, method in enumerate(LONG_RANGE_METHODS):
        subset = city_precision.loc[
            city_precision['DistanceBandAnalysis'].eq(band) & city_precision['Method'].eq(method)
        ]
        target_scatter(axes[row, column], subset,
                       f'{METHOD_LABELS[method]} — {band}', limit=common_limit)
fig.tight_layout()
save_svg(fig, 'rq1_city_direct_landing_scatter.svg')


Exported: figures/rq1_city_direct_landing_error.svg
Exported: figures/rq1_city_direct_landing_scatter.svg


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.8))
for ax, outcome, title, ylabel in (
    (axes[0], 'FirstLandingError2D', 'First landing error', 'Error (m)'),
    (axes[1], 'CorrectionCount', 'Corrective teleport count', 'Corrections'),
):
    sns.boxplot(data=episodes, x='DistanceBandAnalysis', y=outcome, hue='Method',
                order=['200–350 m', '350–500 m'], hue_order=LONG_RANGE_METHODS,
                palette=METHOD_COLORS, showfliers=False, ax=ax)
    sns.stripplot(data=episodes, x='DistanceBandAnalysis', y=outcome, hue='Method',
                  order=['200–350 m', '350–500 m'], hue_order=LONG_RANGE_METHODS,
                  palette=METHOD_COLORS, dodge=True, jitter=.15, alpha=.35,
                  size=2.5, legend=False, ax=ax)
    ax.set(title=title, xlabel='Initial distance', ylabel=ylabel)
axes[0].get_legend().remove()
handles, _ = axes[1].get_legend_handles_labels()
axes[1].get_legend().remove()
fig.legend(handles[:2], [METHOD_LABELS[m] for m in LONG_RANGE_METHODS], title='Method', ncol=2,
           loc='lower center', bbox_to_anchor=(.5, -.01))
fig.tight_layout(rect=(0, .14, 1, 1))
save_svg(fig, 'rq1_checkpoint_approach.svg')

fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.8))
for ax, outcome, title, ylabel in (
    (axes[0], 'FirstLandingError2D', 'First landing error — log scale', 'Error (m)'),
    (axes[1], 'CorrectionCount', 'Corrective teleport count — symlog scale', 'Corrections'),
):
    sns.boxplot(data=episodes, x='DistanceBandAnalysis', y=outcome, hue='Method',
                order=['200–350 m', '350–500 m'], hue_order=LONG_RANGE_METHODS,
                palette=METHOD_COLORS, showfliers=False, ax=ax)
    sns.stripplot(data=episodes, x='DistanceBandAnalysis', y=outcome, hue='Method',
                  order=['200–350 m', '350–500 m'], hue_order=LONG_RANGE_METHODS,
                  palette=METHOD_COLORS, dodge=True, jitter=.15, alpha=.35,
                  size=2.5, legend=False, ax=ax)
    ax.set(title=title, xlabel='Initial distance', ylabel=ylabel)
axes[0].set_yscale('log')
axes[1].set_yscale('symlog', linthresh=1)
axes[0].get_legend().remove()
handles, _ = axes[1].get_legend_handles_labels()
axes[1].get_legend().remove()
fig.legend(handles[:2], [METHOD_LABELS[m] for m in LONG_RANGE_METHODS], title='Method', ncol=2,
           loc='lower center', bbox_to_anchor=(.5, -.01))
fig.tight_layout(rect=(0, .14, 1, 1))
save_svg(fig, 'rq1_checkpoint_approach_log.svg')

fig, axes = plt.subplots(2, 2, figsize=(8.1, 7.4), sharex=True, sharey=True)
common_limit = max(15, float(np.nanquantile(np.abs(episodes[['OffsetX', 'OffsetZ']]), .97)) * 1.08)
for row, band in enumerate(('200–350 m', '350–500 m')):
    for column, method in enumerate(LONG_RANGE_METHODS):
        subset = episodes.loc[
            episodes['DistanceBandAnalysis'].eq(band) & episodes['Method'].eq(method)
        ]
        target_scatter(axes[row, column], subset,
                       f'{METHOD_LABELS[method]} — {band}', limit=common_limit)
fig.tight_layout()
save_svg(fig, 'rq1_checkpoint_approach_scatter.svg')


Exported: figures/rq1_checkpoint_approach.svg
Exported: figures/rq1_checkpoint_approach_log.svg
Exported: figures/rq1_checkpoint_approach_scatter.svg


## RQ2 — Spatial awareness


In [8]:
objective = orientation.groupby(['ParticipantUID', 'Method'], as_index=False)[
    ['BannerDelayTime', 'HeadRotation3D']
].mean()
fig, axes = plt.subplots(1, 4, figsize=(12.2, 3.6))
specs = (
    (objective, 'BannerDelayTime', 'Landing-to-confirmation latency', 'Seconds'),
    (objective, 'HeadRotation3D', 'Head rotation before confirmation', 'Degrees'),
    (spatial, 'SpatialAwarenessComposite', 'Spatial awareness', 'Score (1–6)'),
    (spatial, 'environmental_continuity', 'Environmental continuity', 'Score (1–6)'),
)
for ax, (frame, outcome, title, ylabel) in zip(axes, specs):
    subset = frame.loc[frame['Method'].isin(LONG_RANGE_METHODS)]
    add_box_and_points(ax, subset, 'Method', outcome, LONG_RANGE_METHODS)
    ax.set_xticks(range(2), [METHOD_LABELS[m] for m in LONG_RANGE_METHODS])
    ax.set(title=title, xlabel='', ylabel=ylabel)
fig.tight_layout()
save_svg(fig, 'rq2_reorientation_awareness.svg')


Exported: figures/rq2_reorientation_awareness.svg


In [9]:
def likert_figure(frame, items, title, filename):
    categories = [1, 2, 3, 4, 5, 6]
    colors = {1:'#b2182b', 2:'#ef8a62', 3:'#fddbc7',
              4:'#d1e5f0', 5:'#67a9cf', 6:'#2166ac'}
    rows = []
    for method in METHODS:
        method_data = frame.loc[frame['Method'].eq(method)]
        for column, label in items:
            proportions = method_data[column].value_counts(normalize=True).reindex(categories, fill_value=0) * 100
            rows.append((f'{METHOD_LABELS[method]} — {label}', proportions))
    fig, ax = plt.subplots(figsize=(9.2, max(4.3, .38 * len(rows))))
    for y, (_, proportions) in enumerate(rows):
        left = 0
        for category in (3, 2, 1):
            width = float(proportions[category]); ax.barh(y, -width, left=left,
                color=colors[category], edgecolor='white', linewidth=.4); left -= width
        left = 0
        for category in (4, 5, 6):
            width = float(proportions[category]); ax.barh(y, width, left=left,
                color=colors[category], edgecolor='white', linewidth=.4); left += width
    ax.axvline(0, color='.3', linewidth=.7)
    ax.set_yticks(np.arange(len(rows)), [label for label, _ in rows])
    ax.invert_yaxis(); ax.set_xlabel('Percentage (1–3 disagree; 4–6 agree)')
    ax.set_title(title)
    handles = [plt.Rectangle((0,0),1,1,color=colors[x]) for x in categories]
    ax.legend(handles, [str(x) for x in categories], title='Response', ncol=6,
              loc='upper center', bbox_to_anchor=(.5, -.10))
    fig.tight_layout(rect=(0, .10, 1, 1)); save_svg(fig, filename)

likert_figure(spatial, [
    ('position_awareness', 'Position awareness'),
    ('direction_awareness', 'Direction awareness'),
    ('reorientation_effort', 'Additional reorientation effort'),
    ('environmental_continuity', 'Environmental continuity'),
], 'Spatial-awareness item distributions', 'rq2_spatial_rating_distributions.svg')


Exported: figures/rq2_spatial_rating_distributions.svg


## RQ3 — Usability trade-offs


In [10]:
fig, ax = plt.subplots(figsize=(4.6, 3.8))
add_box_and_points(ax, csq_long, 'Method', 'CSQTotalChange', METHODS)
ax.set_xticks(range(3), [METHOD_LABELS[m] for m in METHODS])
ax.set(title='Cybersickness change', xlabel='', ylabel='CSQ-VR change')
fig.tight_layout()
save_svg(fig, 'rq3_csq_change.svg')

fig, ax = plt.subplots(figsize=(4.6, 3.8))
add_box_and_points(ax, tlx_long, 'Method', 'RAWTLX', METHODS)
ax.set_xticks(range(3), [METHOD_LABELS[m] for m in METHODS])
ax.set(title='Perceived workload', xlabel='', ylabel='RAW-TLX (0–100)')
fig.tight_layout()
save_svg(fig, 'rq3_workload.svg')

labels = ['Pre-experiment', 'After block 1', 'After block 2', 'After block 3']
fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.8), sharey=True)
wide = csq_time.pivot(index='ParticipantUID', columns='Measurement', values='CSQTotal')
for _, values in wide.iterrows():
    axes[0].plot(range(4), values, color='.72', alpha=.55, linewidth=.8)
axes[0].plot(range(4), wide.mean(), marker='o', color='#2166ac', linewidth=2.5, label='Mean')
axes[0].plot(range(4), wide.median(), marker='s', color='#b2182b', linestyle='--', label='Median')
axes[0].legend(loc='upper center', bbox_to_anchor=(.5, -.22), ncol=2)
axes[0].set_ylabel('CSQ-VR Total')
add_box_and_points(axes[1], csq_time, 'Measurement', 'CSQTotal', [0,1,2,3],
                   palette={0:'#9ecae1',1:'#9ecae1',2:'#9ecae1',3:'#9ecae1'})
for ax in axes:
    ax.set_xticks(range(4), labels, rotation=18, ha='right'); ax.set_xlabel('Measurement')
axes[0].set_title('Individual trajectories'); axes[1].set_title('Measurement distributions')
fig.tight_layout(rect=(0, .16, 1, 1))
save_svg(fig, 'rq3_csq_timecourse.svg')


Exported: figures/rq3_csq_change.svg
Exported: figures/rq3_workload.svg
Exported: figures/rq3_csq_timecourse.svg


In [11]:
likert_figure(spatial, [
    ('environment_distance', 'Environmental distance awareness'),
    ('destination_distance', 'Destination distance awareness'),
    ('easy_to_learn', 'Easy to learn'),
    ('intuitive', 'Intuitive to use'),
], 'Usability and distance-awareness item distributions',
   'rq3_usability_rating_distributions.svg')

ranking_prefixes = ['Overall Preference for Long-Range Navigation', 'Comfort During Use',
                    'Perceived Intuitiveness', 'Fun Factor']
rank_categories = ['1st', '2nd', '3rd']
rows = []
for question in ranking_prefixes:
    for method in METHODS:
        column = f'{question} [{METHOD_LABELS[method]}]'
        counts = preferences[column].value_counts().reindex(rank_categories, fill_value=0)
        rows.append({'Label': f'{question} — {METHOD_LABELS[method]}',
                     **{rank: int(counts[rank]) for rank in rank_categories}})
ranking = pd.DataFrame(rows)
colors = {'1st':'#2ca25f', '2nd':'#99d8c9', '3rd':'#de2d26'}
fig, ax = plt.subplots(figsize=(9.8, 6.2))
y = np.arange(len(ranking)); left = np.zeros(len(ranking))
totals = ranking[rank_categories].sum(axis=1).to_numpy(float)
for rank in rank_categories:
    values = ranking[rank].to_numpy(float) / totals * 100
    ax.barh(y, values, left=left, color=colors[rank], label=rank,
            edgecolor='white', linewidth=.4); left += values
ax.set_yticks(y, ranking['Label']); ax.invert_yaxis(); ax.set_xlim(0,100)
ax.set_xlabel('Percentage of participants'); ax.set_title('Preference rankings')
ax.legend(title='Rank', ncol=3, loc='upper center', bbox_to_anchor=(.5, -.10))
fig.tight_layout(rect=(0, .10, 1, 1))
save_svg(fig, 'rq3_preference_rankings.svg')


Exported: figures/rq3_usability_rating_distributions.svg
Exported: figures/rq3_preference_rankings.svg


## Export manifest


In [12]:
manifest = pd.DataFrame({
    'Figure': [path.name for path in EXPORTED_FIGURES],
    'RelativePath': [str(path) for path in EXPORTED_FIGURES],
    'Bytes': [path.stat().st_size for path in EXPORTED_FIGURES],
})
display(manifest)
assert len(manifest) == 17
assert manifest['Bytes'].gt(0).all()
print(f'Export complete: {len(manifest)} SVG figures.')


                                    Figure  ...   Bytes
0       rq1_city_race_completion_times.svg  ...  164489
1           rq1_interaction_efficiency.svg  ...   57324
2          rq1_precision_landing_error.svg  ...  200225
3                 rq1_precision_timing.svg  ...  398731
4           rq1_precision_200m_scatter.svg  ...   76601
5        rq1_city_direct_landing_error.svg  ...  147662
6      rq1_city_direct_landing_scatter.svg  ...  182188
7              rq1_checkpoint_approach.svg  ...  507760
8          rq1_checkpoint_approach_log.svg  ...  505018
9      rq1_checkpoint_approach_scatter.svg  ...  276512
10         rq2_reorientation_awareness.svg  ...   85873
11    rq2_spatial_rating_distributions.svg  ...   35364
12                      rq3_csq_change.svg  ...   30990
13                        rq3_workload.svg  ...   30477
14                  rq3_csq_timecourse.svg  ...   57610
15  rq3_usability_rating_distributions.svg  ...   35315
16             rq3_preference_rankings.svg  ... 